# 02 - Vintage trends and time-series modeling

The monthly default rate by origination vintage is the aggregate credit-cycle
signal in this dataset. This notebook decomposes that series, tests it for
stationarity (ADF and KPSS) and fits a SARIMAX model with a chronological
train/test split, checking the residuals with a Ljung-Box test.

In [ ]:
import pathlib, sys
ROOT = pathlib.Path.cwd()
while not (ROOT / 'config' / 'config.yaml').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.width', 160)

In [ ]:
from src.config import Config
from src.pipeline import load_modeling_frame
from src import timeseries

cfg = Config.load()
df = load_modeling_frame(cfg)
vintage = timeseries.vintage_table(df)
display(vintage.head())
display(timeseries.monthly_default_series(df).describe().round(4))

In [ ]:
out = timeseries.run_timeseries(df)
display(out['stationarity'].round(4))
display(pd.Series(out['metrics']))
display(out['ljung_box'])

**Reading the tests.** ADF fails to reject the unit root and KPSS rejects
stationarity, so the level series is non-stationary; the SARIMAX model is
therefore fitted with one order of differencing (`d = 1`). The Ljung-Box test
on the residuals is a diagnostic of whether the fitted model captured the
autocorrelation: a small p-value means structure remains in the residuals.

In [ ]:
for name in ['ts_decomposition.png', 'acf_pacf.png', 'ts_forecast.png']:
    display(Image(filename=str(ROOT / 'reports' / 'figures' / name)))

In [ ]:
display(timeseries.vintage_table(df).tail(15))

### Caveats

* Vintages post-2018 are immature: only resolved loans enter the series, so
  recent months are mechanically under-represented (survivorship in the
  label). The pipeline drops months with fewer than 200 resolved loans.
* The series is an **unweighted** mean of monthly default rates; a
  volume-weighted series is also available from `timeseries.vintage_table`.